In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_bronze = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("Files/bronze/shipments_raw.csv")

print(f"Bronze rows: {df_bronze.count()}")
df_bronze.printSchema()

StatementMeta(, 36250b65-bb1b-45f3-a5e0-280436a7cbb1, 3, Finished, Available, Finished, False)

Bronze rows: 10000
root
 |-- shipment_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- carrier_name: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- on_time_flag: integer (nullable = true)
 |-- weight_lbs: double (nullable = true)
 |-- distance_miles: integer (nullable = true)
 |-- promised_transit_days: integer (nullable = true)
 |-- actual_transit_days: double (nullable = true)
 |-- freight_cost_usd: double (nullable = true)
 |-- freight_revenue_usd: double (nullable = true)
 |-- has_claim: integer (nullable = true)
 |-- claim_amount_usd: double (nullable = true)



In [2]:
df_carriers_raw = spark.read.option("multiline", True).json("Files/bronze/carrier_master.json")
df_carriers = df_carriers_raw.select(explode(col("carriers")).alias("c")).select("c.*")
df_carriers.show()

StatementMeta(, 36250b65-bb1b-45f3-a5e0-280436a7cbb1, 5, Finished, Available, Finished, False)

+----------+-------------+----------+--------------+------------+
|carrier_id| carrier_name|fleet_size|       hq_city|sla_otdr_pct|
+----------+-------------+----------+--------------+------------+
|      C001|XPO Logistics|     38000|  Greenwich CT|          92|
|      C002|Estes Express|      7000|   Richmond VA|          90|
|      C003| Old Dominion|     10000|Thomasville NC|          95|
|      C004|  ABF Freight|      4500| Fort Smith AR|          88|
|      C005|     Saia LTL|      5000|Johns Creek GA|          90|
+----------+-------------+----------+--------------+------------+



In [3]:
df_silver = df_bronze \
    .withColumn("pickup_date",            to_date("pickup_date", "yyyy-MM-dd")) \
    .withColumn("promised_delivery_date", to_date("promised_delivery_date", "yyyy-MM-dd")) \
    .withColumn("actual_delivery_date",   to_date("actual_delivery_date", "yyyy-MM-dd")) \
    .withColumn("transit_variance_days",
                col("actual_transit_days") - col("promised_transit_days")) \
    .withColumn("gross_margin_usd",
                round(col("freight_revenue_usd") - col("freight_cost_usd"), 2)) \
    .withColumn("gross_margin_pct",
                round((col("gross_margin_usd") / col("freight_revenue_usd")) * 100, 2)) \
    .withColumn("cost_per_mile",
                round(col("freight_cost_usd") / col("distance_miles"), 4)) \
    .withColumn("revenue_per_mile",
                round(col("freight_revenue_usd") / col("distance_miles"), 4)) \
    .withColumn("pickup_month",   month("pickup_date")) \
    .withColumn("pickup_quarter", quarter("pickup_date")) \
    .withColumn("pickup_year",    year("pickup_date")) \
    .withColumn("lane", concat_ws(" -> ", col("origin_city"), col("destination_city"))) \
    .dropDuplicates(["shipment_id"]) \
    .filter(col("shipment_id").isNotNull())

# Join carrier SLA data
df_silver = df_silver.join(
    df_carriers.select("carrier_name", "carrier_id", "sla_otdr_pct"),
    on="carrier_name",
    how="left"
)

print(f"Silver rows: {df_silver.count()}")

StatementMeta(, 36250b65-bb1b-45f3-a5e0-280436a7cbb1, 7, Finished, Available, Finished, False)

Silver rows: 10000


In [4]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable("silver_shipments")

print("silver_shipments Delta table created.")

StatementMeta(, 36250b65-bb1b-45f3-a5e0-280436a7cbb1, 8, Finished, Available, Finished, False)

silver_shipments Delta table created.
